# LeetCode 778 — Swim in Rising Water bằng Python

- Link: https://leetcode.com/problems/swim-in-rising-water/
- Độ khó: **Hard**
- Chủ đề: Đồ thị lưới, Tìm kiếm nhị phân trên đáp án, BFS/DFS, Dijkstra (đường đi minimax), Union-Find (Kruskal)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def swimInWater(self, grid: List[List[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Bạn đứng ở ô trên-trái của một lưới độ cao `n × n`. Mưa làm mực nước dâng lên: ở thời điểm `t`, mực nước là `t`. Bạn có thể bơi từ ô này sang ô kề cạnh **ngay lập tức** nếu **cả hai ô** đều có độ cao `≤ t`. Hỏi thời điểm **sớm nhất** để tới ô dưới-phải.

Đây là bài kinh điển của dạng **"đường đi minimax"** (bottleneck path): trong tất cả các đường đi, ta muốn đường có **giá trị lớn nhất trên đường** là **nhỏ nhất**. Cùng một bài có thể giải bằng 4 tư duy khác nhau, rất đáng học.

Sau notebook này bạn sẽ:

1. Nhận ra bài toán "nhỏ nhất của lớn nhất trên đường đi" (minimax path).
2. Dùng **tìm kiếm nhị phân trên đáp án** kết hợp BFS/DFS kiểm tra "có đi được không".
3. Sửa **Dijkstra** để chi phí đường đi là `max` thay vì tổng.
4. Dùng **Union-Find** bật ô theo thứ tự độ cao tăng dần (tư duy Kruskal).
5. Chứng minh vì sao mỗi cách đúng và so sánh độ phức tạp.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho lưới `grid` kích thước `n × n`, `grid[i][j]` là độ cao ô `(i, j)`. Các giá trị là **một hoán vị** của `0, 1, ..., n²-1` (đôi một khác nhau). Ở thời điểm `t`, độ sâu nước mọi nơi là `t`. Bạn bơi từ ô sang ô kề 4 hướng khi và chỉ khi độ cao **của cả hai ô** `≤ t`. Bơi không tốn thời gian, nhưng phải ở trong lưới. Bắt đầu ở `(0, 0)`; trả về thời gian nhỏ nhất để tới `(n-1, n-1)`.

**Ví dụ 1**

```text
Input: grid = [[0,2],[1,3]]
Output: 3
Giải thích: ở t = 0 bạn ở (0,0). Không đi được đâu vì các ô kề có độ cao 1 và 2 > 0.
Phải đợi tới t = 3 thì ô đích (độ cao 3) mới ngập.
```

**Ví dụ 2**

```text
Input: grid = [[0,1,2,3,4],[24,23,22,21,5],[12,13,14,15,16],[11,17,18,19,20],[10,9,8,7,6]]
Output: 16
Giải thích: đi theo hàng đầu, cột cuối xuống tới 16 rồi tiếp; ta buộc phải đi qua một ô cao 16.
```

**Ràng buộc**

- `n == grid.length == grid[i].length`
- `1 <= n <= 50`
- `0 <= grid[i][j] < n²`
- Mỗi giá trị `grid[i][j]` là **duy nhất**.

In [1]:
from typing import List

EXAMPLES = [
    ([[0, 2], [1, 3]], 3),
    ([[0, 1, 2, 3, 4], [24, 23, 22, 21, 5], [12, 13, 14, 15, 16],
      [11, 17, 18, 19, 20], [10, 9, 8, 7, 6]], 16),
    ([[0]], 0),
    ([[3, 2], [0, 1]], 3),
]
for g, exp in EXAMPLES:
    print(len(g), "x", len(g), "-> kỳ vọng", exp)

2 x 2 -> kỳ vọng 3
5 x 5 -> kỳ vọng 16
1 x 1 -> kỳ vọng 0
2 x 2 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Duyệt 4 hướng trên lưới

Một ô `(r, c)` có tối đa 4 láng giềng `(r±1, c)`, `(r, c±1)`. Mẫu chuẩn: một danh sách hướng `DIRS` và kiểm tra biên `0 <= nr < n and 0 <= nc < n`.

In [2]:
DIRS = ((1, 0), (-1, 0), (0, 1), (0, -1))

def neighbors(r, c, n):
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < n and 0 <= nc < n:
            yield nr, nc

print(list(neighbors(0, 0, 3)))
print(list(neighbors(1, 1, 3)))

[(1, 0), (0, 1)]
[(2, 1), (0, 1), (1, 2), (1, 0)]


### 3.2. BFS với `deque`

BFS từ một ô nguồn, chỉ đi vào những ô thoả điều kiện. Dùng `collections.deque` để `popleft()` là `O(1)` (khác với `list.pop(0)` là `O(n)`).

In [3]:
from collections import deque

def bfs_reach(grid, allowed):
    """Trả về tập các ô tới được từ (0,0) chỉ đi qua ô có allowed(h) đúng."""
    n = len(grid)
    if not allowed(grid[0][0]):
        return set()
    seen = {(0, 0)}
    q = deque([(0, 0)])
    while q:
        r, c = q.popleft()
        for nr, nc in neighbors(r, c, n):
            if (nr, nc) not in seen and allowed(grid[nr][nc]):
                seen.add((nr, nc))
                q.append((nr, nc))
    return seen

g = [[0, 2], [1, 3]]
print(bfs_reach(g, lambda h: h <= 1))
print(bfs_reach(g, lambda h: h <= 3))

{(1, 0), (0, 0)}
{(1, 0), (0, 1), (1, 1), (0, 0)}


### 3.3. Tìm kiếm nhị phân trên đáp án

Nếu có một vị từ `ok(t)` **đơn điệu** (sai, sai, ..., sai, đúng, đúng, ...), ta tìm `t` nhỏ nhất làm `ok(t)` đúng bằng nhị phân: `lo, hi`; nếu `ok(mid)` đúng thì `hi = mid`, ngược lại `lo = mid + 1`.

In [4]:
def first_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi) // 2
        if ok(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

# ví dụ: số nhỏ nhất có bình phương >= 50
print(first_true(0, 100, lambda x: x * x >= 50))

8


### 3.4. `heapq` — hàng đợi ưu tiên

`heapq.heappush(h, (key, ...))` và `heapq.heappop(h)` luôn lấy ra phần tử có `key` nhỏ nhất, mỗi thao tác `O(log size)`.

In [5]:
import heapq
h = []
for item in [(5, 'a'), (1, 'b'), (3, 'c')]:
    heapq.heappush(h, item)
print([heapq.heappop(h) for _ in range(3)])

[(1, 'b'), (3, 'c'), (5, 'a')]


### 3.5. Union-Find (Disjoint Set Union)

Cấu trúc gộp tập hợp: `find(x)` trả về "đại diện" của tập chứa `x`, `union(a, b)` gộp hai tập. Với **nén đường đi** + **gộp theo kích thước**, mỗi thao tác gần như `O(1)`.

In [6]:
class DSU:
    def __init__(self, n):
        self.p = list(range(n))
        self.sz = [1] * n

    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]   # nén đường đi (halving)
            x = self.p[x]
        return x

    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a == b:
            return False
        if self.sz[a] < self.sz[b]:
            a, b = b, a
        self.p[b] = a
        self.sz[a] += self.sz[b]
        return True

d = DSU(5)
d.union(0, 1); d.union(3, 4)
print(d.find(1) == d.find(0), d.find(1) == d.find(3))

True False


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Diễn giải lại

Ở thời điểm `t`, các ô "ngập" là ô có độ cao `≤ t`. Ta đi được giữa hai ô ngập kề nhau. Vậy câu hỏi: **`t` nhỏ nhất** sao cho `(0,0)` và `(n-1,n-1)` **cùng thuộc một vùng liên thông** của các ô ngập.

Một cách nhìn khác: mỗi đường đi `P` từ đầu tới đích cần đợi tới thời điểm `max(grid[ô] for ô in P)`. Đáp án là

```text
min over all paths P of  max over cells in P of grid[cell]
```

→ **minimax path**.

### 4.2. Tính tay ví dụ 1

```text
0 2
1 3
```

Có 2 đường đi: `0 → 2 → 3` (max 3) và `0 → 1 → 3` (max 3). Đáp án `3`. Chú ý ô đích luôn nằm trên đường nên đáp án `≥ grid[n-1][n-1]`, và tương tự `≥ grid[0][0]`.

### 4.3. Tính tay một lưới 3×3

```text
0 5 2
6 4 8
1 3 7
```

- Đi phải-xuống: `0 → 5 → 4 → 3 → 7` có max 7? Ô `(1,1)=4`, `(2,1)=3`, `(2,2)=7` → max = `max(0,5,4,3,7) = 7`.
- Đi xuống trước: `0 → 6 → 1 → 3 → 7` → max 7.
- Đáp án không thể nhỏ hơn 7 vì ô đích cao 7. Vậy đáp án = **7**.

### 4.4. Tính đơn điệu

Nếu tới được đích ở thời điểm `t`, thì cũng tới được ở mọi `t' > t` (nước chỉ dâng, ô ngập vẫn ngập). → vị từ `can(t)` đơn điệu → tìm kiếm nhị phân được.

In [7]:
g3 = [[0, 5, 2], [6, 4, 8], [1, 3, 7]]
n3 = len(g3)
for t in range(9):
    reach = bfs_reach(g3, lambda h, t=t: h <= t)
    print(f"t={t}: tới đích? {(n3 - 1, n3 - 1) in reach}  | số ô tới được = {len(reach)}")

t=0: tới đích? False  | số ô tới được = 1
t=1: tới đích? False  | số ô tới được = 1
t=2: tới đích? False  | số ô tới được = 1
t=3: tới đích? False  | số ô tới được = 1
t=4: tới đích? False  | số ô tới được = 1
t=5: tới đích? False  | số ô tới được = 6
t=6: tới đích? False  | số ô tới được = 7
t=7: tới đích? True  | số ô tới được = 8
t=8: tới đích? True  | số ô tới được = 9


## 5. Cách 1 — Brute Force: thử từng thời điểm `t`

**Ý tưởng.** Thử `t = 0, 1, 2, ...` theo thứ tự. Với mỗi `t`, chạy BFS từ `(0,0)` chỉ qua các ô `≤ t`. Lần đầu tới được đích → trả `t`.

Đáp án tối đa là `n² - 1` (khi đó mọi ô đều ngập), nên vòng lặp chắc chắn dừng.

In [8]:
def can_reach(grid, t):
    n = len(grid)
    return (n - 1, n - 1) in bfs_reach(grid, lambda h: h <= t)

def swim_brute(grid):
    n = len(grid)
    for t in range(n * n):
        if can_reach(grid, t):
            return t
    return n * n - 1

for g, exp in EXAMPLES:
    print(swim_brute(g), "(kỳ vọng", exp, ")")
    assert swim_brute(g) == exp

3 (kỳ vọng 3 )
16 (kỳ vọng 16 )
0 (kỳ vọng 0 )
3 (kỳ vọng 3 )


**Độ phức tạp.** Có tối đa `n²` giá trị `t`, mỗi lần BFS `O(n²)` → **`O(n⁴)`** thời gian, `O(n²)` bộ nhớ. Với `n = 50`: `n⁴ = 6.25 triệu` lần thăm ô — vẫn chạy được trong C++ nhưng chậm trong Python (mỗi ô cũng tốn vài phép tính).

**Nút thắt.** Ta thử `t` **tuần tự** dù vị từ đơn điệu. Hai hướng cải tiến:

1. Thay vì tuần tự, **nhị phân** trên `t` → `O(n² log n²)`.
2. Không làm lại từ đầu cho mỗi `t`: **mở rộng dần** vùng ngập (Dijkstra / Union-Find).

## 6. Bước đột phá

**Insight 1 — Đơn điệu.** `can(t)` sai rồi đúng mãi mãi. → *Nếu* vị từ đơn điệu *thì* tìm kiếm nhị phân trên đáp án.

**Insight 2 — Chi phí đường đi là `max`.** Định nghĩa `cost(P) = max độ cao trên P`. Hàm `max` cũng "đơn điệu" như phép cộng với trọng số không âm: kéo dài đường đi thì `cost` không giảm. → *Nếu* chi phí ghép bằng phép toán không giảm (`+` với số không âm, `max`, ...) *thì* **Dijkstra** vẫn đúng. Ở đây khoảng cách của ô láng giềng là `max(dist[u], grid[v])`.

**Insight 3 — Bật ô theo độ cao tăng dần.** Xem quá trình nước dâng: ở thời điểm `t`, đúng một ô mới được ngập (ô có độ cao `t`, vì các độ cao khác nhau). Gộp ô đó với các láng giềng đã ngập. Lần đầu `(0,0)` và `(n-1,n-1)` cùng tập → đáp án `t`. → **Union-Find** (giống Kruskal).

**Invariant chung:** ở mọi thời điểm, "tập ô tới được từ đầu" chính là thành phần liên thông chứa `(0,0)` trong đồ thị con các ô `≤ t`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tìm kiếm nhị phân + BFS

- `lo = max(grid[0][0], grid[n-1][n-1])` (cận dưới hiển nhiên), `hi = n² - 1`.
- Nhị phân tìm `t` nhỏ nhất mà `can_reach(grid, t)` đúng.

In [9]:
def swim_binary(grid):
    n = len(grid)
    lo, hi = max(grid[0][0], grid[-1][-1]), n * n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if can_reach(grid, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

for g, exp in EXAMPLES:
    assert swim_binary(g) == exp
print("Binary search + BFS: OK")

Binary search + BFS: OK


Code nộp LeetCode cho cách 2 (viết BFS trực tiếp, dùng mảng `seen` 2 chiều cho nhanh):

In [10]:
class SolutionBinary:
    def swimInWater(self, grid: List[List[int]]) -> int:
        n = len(grid)

        def ok(t):
            if grid[0][0] > t:
                return False
            seen = [[False] * n for _ in range(n)]
            seen[0][0] = True
            stack = [(0, 0)]
            while stack:
                r, c = stack.pop()
                if r == n - 1 and c == n - 1:
                    return True
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < n and 0 <= nc < n and not seen[nr][nc] and grid[nr][nc] <= t:
                        seen[nr][nc] = True
                        stack.append((nr, nc))
            return False

        lo, hi = max(grid[0][0], grid[-1][-1]), n * n - 1
        while lo < hi:
            mid = (lo + hi) // 2
            if ok(mid):
                hi = mid
            else:
                lo = mid + 1
        return lo

for g, exp in EXAMPLES:
    assert SolutionBinary().swimInWater(g) == exp
print("SolutionBinary: OK")

SolutionBinary: OK


**Độ phức tạp cách 2.** `O(log(n²))` lần kiểm tra, mỗi lần `O(n²)` → **`O(n² log n)`** thời gian, `O(n²)` bộ nhớ. (Ở đây dùng DFS bằng stack cũng được — chỉ cần biết *có* tới được hay không, không cần khoảng cách.)

### 7.2. Cách 3 — Dijkstra với chi phí `max`

- `dist[v]` = thời điểm sớm nhất để tới `v` (= min trên các đường của max độ cao).
- Khởi tạo `dist[(0,0)] = grid[0][0]`; heap chứa `(dist, r, c)`.
- Lấy ô có `dist` nhỏ nhất; với láng giềng `v`: `cand = max(dist[u], grid[v])`; nếu tốt hơn thì cập nhật.
- Khi lấy ra đích → trả về `dist`.

Thực ra vì chi phí `max` và mỗi ô chỉ được "chốt" một lần, ta có thể đơn giản hoá: đánh dấu `seen` khi đẩy vào heap (vì khi đẩy, `cand` đã là tốt nhất có thể? — không hẳn! xem phần chứng minh mục 10). Cách an toàn nhất là Dijkstra chuẩn.

In [11]:
def swim_dijkstra(grid):
    n = len(grid)
    INF = float('inf')
    dist = [[INF] * n for _ in range(n)]
    dist[0][0] = grid[0][0]
    heap = [(grid[0][0], 0, 0)]
    while heap:
        d, r, c = heapq.heappop(heap)
        if d > dist[r][c]:
            continue                      # bản ghi cũ
        if r == n - 1 and c == n - 1:
            return d
        for nr, nc in neighbors(r, c, n):
            cand = max(d, grid[nr][nc])
            if cand < dist[nr][nc]:
                dist[nr][nc] = cand
                heapq.heappush(heap, (cand, nr, nc))
    return dist[n - 1][n - 1]

for g, exp in EXAMPLES:
    assert swim_dijkstra(g) == exp
print("Dijkstra (max): OK")

Dijkstra (max): OK


**Phiên bản gọn hay gặp.** Với bài này, đánh dấu `seen` ngay khi đẩy vào heap cũng đúng. Lý do: heap lấy ra theo `max` tăng dần; ô `v` được đẩy lần đầu với `cand = max(d_u, grid[v])`, và mọi đường khác tới `v` đều đi qua một ô được lấy ra **sau** `u` (nên có `d ≥ d_u`) và phải chứa `grid[v]` → không tốt hơn. Đây là code nộp phổ biến nhất:

In [12]:
class SolutionDijkstra:
    def swimInWater(self, grid: List[List[int]]) -> int:
        n = len(grid)
        seen = [[False] * n for _ in range(n)]
        seen[0][0] = True
        heap = [(grid[0][0], 0, 0)]
        ans = 0
        while heap:
            h, r, c = heapq.heappop(heap)
            ans = max(ans, h)          # mực nước phải ít nhất bằng ô vừa bước vào
            if r == n - 1 and c == n - 1:
                return ans
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and not seen[nr][nc]:
                    seen[nr][nc] = True
                    heapq.heappush(heap, (grid[nr][nc], nr, nc))
        return ans

for g, exp in EXAMPLES:
    assert SolutionDijkstra().swimInWater(g) == exp
print("SolutionDijkstra: OK")

SolutionDijkstra: OK


Phiên bản gọn này còn có cách hiểu trực quan là **"loang nước theo ô thấp nhất"**: tập ô đã chiếm là một vùng liên thông chứa `(0,0)`; mỗi bước ta luôn bước vào ô **thấp nhất** trên biên. `ans` = độ cao lớn nhất đã phải bước vào = mực nước cần thiết.

**Độ phức tạp cách 3.** Mỗi ô vào heap tối đa 1 lần (bản gọn) hoặc vài lần (bản chuẩn) → **`O(n² log n)`** thời gian, `O(n²)` bộ nhớ.

### 7.3. Cách 4 — Union-Find theo thứ tự độ cao (Kruskal)

- Vì các độ cao là hoán vị của `0..n²-1`, lập `pos[h] = (r, c)`.
- Duyệt `t = 0, 1, 2, ...`: bật ô `pos[t]`, gộp nó với các láng giềng **đã bật** (độ cao `< t`).
- Sau mỗi bước, nếu `find(0) == find(n²-1)` → trả `t`.

In [13]:
def swim_dsu(grid):
    n = len(grid)
    N = n * n
    pos = [None] * N
    for r in range(n):
        for c in range(n):
            pos[grid[r][c]] = (r, c)
    dsu = DSU(N)
    for t in range(N):
        r, c = pos[t]
        for nr, nc in neighbors(r, c, n):
            if grid[nr][nc] < t:          # láng giềng đã ngập
                dsu.union(r * n + c, nr * n + nc)
        if dsu.find(0) == dsu.find(N - 1):
            return t
    return N - 1

for g, exp in EXAMPLES:
    assert swim_dsu(g) == exp
print("Union-Find: OK")

Union-Find: OK


Code nộp LeetCode cho cách 4:

In [14]:
class SolutionDSU:
    def swimInWater(self, grid: List[List[int]]) -> int:
        n = len(grid)
        N = n * n
        parent = list(range(N))

        def find(x):
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        pos = [0] * N
        for r in range(n):
            for c in range(n):
                pos[grid[r][c]] = r * n + c
        for t in range(N):
            idx = pos[t]
            r, c = divmod(idx, n)
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] < t:
                    a, b = find(idx), find(nr * n + nc)
                    if a != b:
                        parent[a] = b
            if find(0) == find(N - 1):
                return t
        return N - 1

for g, exp in EXAMPLES:
    assert SolutionDSU().swimInWater(g) == exp
print("SolutionDSU: OK")

SolutionDSU: OK


**Độ phức tạp cách 4.** Lập `pos` `O(n²)`; mỗi ô gộp với ≤ 4 láng giềng, mỗi `find` gần `O(1)` (α(n²)) → **`O(n² α(n²))`** ≈ tuyến tính. Bộ nhớ `O(n²)`. Không cần sắp xếp nhờ các độ cao là hoán vị (nếu không, phải sort → `O(n² log n)`).

## 8. TRACE từng bước

### 8.1. Trace cách 3 (bản "loang ô thấp nhất") trên lưới 3×3

```text
0 5 2
6 4 8
1 3 7
```

In [15]:
def trace_dijkstra(grid):
    n = len(grid)
    seen = {(0, 0)}
    heap = [(grid[0][0], 0, 0)]
    ans, step = 0, 0
    print(f"{'bước':>4} | {'lấy ra':>10} | {'ans':>3} | heap sau khi đẩy")
    while heap:
        h, r, c = heapq.heappop(heap)
        ans = max(ans, h)
        pushed = []
        if (r, c) != (n - 1, n - 1):
            for nr, nc in neighbors(r, c, n):
                if (nr, nc) not in seen:
                    seen.add((nr, nc))
                    heapq.heappush(heap, (grid[nr][nc], nr, nc))
                    pushed.append(grid[nr][nc])
        print(f"{step:>4} | h={h} @({r},{c}) | {ans:>3} | đẩy {pushed}, heap={sorted(x[0] for x in heap)}")
        step += 1
        if (r, c) == (n - 1, n - 1):
            return ans

print("Đáp án:", trace_dijkstra(g3))

bước |     lấy ra | ans | heap sau khi đẩy
   0 | h=0 @(0,0) |   0 | đẩy [6, 5], heap=[5, 6]
   1 | h=5 @(0,1) |   5 | đẩy [4, 2], heap=[2, 4, 6]
   2 | h=2 @(0,2) |   5 | đẩy [8], heap=[4, 6, 8]
   3 | h=4 @(1,1) |   5 | đẩy [3], heap=[3, 6, 8]
   4 | h=3 @(2,1) |   5 | đẩy [7, 1], heap=[1, 6, 7, 8]
   5 | h=1 @(2,0) |   5 | đẩy [], heap=[6, 7, 8]
   6 | h=6 @(1,0) |   6 | đẩy [], heap=[7, 8]
   7 | h=7 @(2,2) |   7 | đẩy [], heap=[8]
Đáp án: 7


Bảng trạng thái (đọc từ output):

| bước | ô lấy ra (độ cao) | ans | ghi chú |
|---|---|---|---|
| 0 | (0,0)=0 | 0 | đẩy 5 và 6 |
| 1 | (0,1)=5 | 5 | buộc phải lên 5 vì cả hai cửa ra (5, 6) đều ≥ 5 |
| 2 | (1,1)=4 | 5 | 4 < ans nên ans không đổi |
| 3 | (2,1)=3 | 5 | |
| 4 | (2,0)=1 hoặc (0,2)=2 ... | 5 | các ô thấp được "tràn" trước |
| ... | (2,2)=7 | 7 | tới đích |

### 8.2. Trace cách 4 (Union-Find) — nước dâng từng mức

In [16]:
def trace_dsu(grid):
    n = len(grid)
    N = n * n
    pos = {grid[r][c]: (r, c) for r in range(n) for c in range(n)}
    dsu = DSU(N)
    for t in range(N):
        r, c = pos[t]
        merged = []
        for nr, nc in neighbors(r, c, n):
            if grid[nr][nc] < t and dsu.union(r * n + c, nr * n + nc):
                merged.append(grid[nr][nc])
        # vẽ lưới: '#' = đã ngập, '.' = khô
        rows = [''.join('#' if grid[i][j] <= t else '.' for j in range(n)) for i in range(n)]
        conn = dsu.find(0) == dsu.find(N - 1)
        print(f"t={t}: bật ô ({r},{c}), gộp với {merged}; lưới {rows}; nối? {conn}")
        if conn:
            return t

print("Đáp án:", trace_dsu(g3))

t=0: bật ô (0,0), gộp với []; lưới ['#..', '...', '...']; nối? False
t=1: bật ô (2,0), gộp với []; lưới ['#..', '...', '#..']; nối? False
t=2: bật ô (0,2), gộp với []; lưới ['#.#', '...', '#..']; nối? False
t=3: bật ô (2,1), gộp với [1]; lưới ['#.#', '...', '##.']; nối? False
t=4: bật ô (1,1), gộp với [3]; lưới ['#.#', '.#.', '##.']; nối? False
t=5: bật ô (0,1), gộp với [4, 2, 0]; lưới ['###', '.#.', '##.']; nối? False
t=6: bật ô (1,0), gộp với [1]; lưới ['###', '##.', '##.']; nối? False
t=7: bật ô (2,2), gộp với [3]; lưới ['###', '##.', '###']; nối? True
Đáp án: 7


Nhận xét: ở `t = 6`, các ô ngập là `{0,1,2,3,4,5,6}` — gần như cả lưới, nhưng ô đích (7) vẫn khô. Đến `t = 7` ô đích ngập và nối vào vùng lớn → đáp án 7.

### 8.3. Trace cách 2 (nhị phân) trên ví dụ 2

In [17]:
def trace_binary(grid):
    n = len(grid)
    lo, hi = max(grid[0][0], grid[-1][-1]), n * n - 1
    while lo < hi:
        mid = (lo + hi) // 2
        res = can_reach(grid, mid)
        print(f"lo={lo:2} hi={hi:2} mid={mid:2} can_reach={res}")
        if res:
            hi = mid
        else:
            lo = mid + 1
    print("=> đáp án", lo)
    return lo

trace_binary(EXAMPLES[1][0])

lo= 6 hi=24 mid=15 can_reach=False
lo=16 hi=24 mid=20 can_reach=True
lo=16 hi=20 mid=18 can_reach=True
lo=16 hi=18 mid=17 can_reach=True
lo=16 hi=17 mid=16 can_reach=True
=> đáp án 16


16

## 9. Code nộp LeetCode

Chọn cách 3 (bản gọn) — ngắn, nhanh, dễ nhớ:

In [18]:
import heapq
from typing import List

class Solution:
    def swimInWater(self, grid: List[List[int]]) -> int:
        n = len(grid)
        seen = [[False] * n for _ in range(n)]
        seen[0][0] = True
        heap = [(grid[0][0], 0, 0)]
        ans = 0
        while heap:
            h, r, c = heapq.heappop(heap)
            if h > ans:
                ans = h
            if r == n - 1 and c == n - 1:
                return ans
            for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                if 0 <= nr < n and 0 <= nc < n and not seen[nr][nc]:
                    seen[nr][nc] = True
                    heapq.heappush(heap, (grid[nr][nc], nr, nc))
        return ans

for g, exp in EXAMPLES:
    assert Solution().swimInWater(g) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Định nghĩa.** `OPT = min_P max_{x∈P} grid[x]`, `can(t)` = "đầu và đích liên thông trong đồ thị các ô `≤ t`". Ta có `can(t) ⇔ t ≥ OPT`:

- Nếu `t ≥ OPT`: đường tối ưu có mọi ô `≤ OPT ≤ t` → liên thông.
- Nếu `can(t)`: có đường mọi ô `≤ t` → `OPT ≤ t`.

**Cách 1, 2** trả về `t` nhỏ nhất có `can(t)` → đúng bằng `OPT`. Cách 2 đúng vì `can` đơn điệu.

**Cách 4.** Sau bước `t`, DSU biểu diễn đúng các thành phần liên thông của đồ thị con các ô `≤ t` (mỗi cạnh giữa hai ô `≤ t` được xét khi ô cao hơn được bật). Vậy lần đầu đầu–đích cùng tập là `t` nhỏ nhất có `can(t)`.

**Cách 3 (bản gọn).** Gọi `S` là tập ô đã lấy ra. Bổ đề: khi ô `x` được lấy ra với `ans` hiện tại, mọi ô trong `S` đều tới được từ đầu với mực nước `ans`, và **không** có cách nào thoát khỏi `S` với mực nước `< ` (độ cao nhỏ nhất trên biên). Vì vậy lần đầu `ans` tăng lên giá trị `h`, mọi đường tới phần ngoài `S` đều phải đi qua một ô biên có độ cao `≥ h` → không thể làm tốt hơn. Khi đích được lấy ra, `ans` là mực nước nhỏ nhất cần thiết. (Đây cũng chính là thuật toán Prim xây cây khung nhỏ nhất theo đỉnh — trên cây khung nhỏ nhất, đường đi giữa hai đỉnh là đường minimax.)

**Cách 3 (bản chuẩn)** là Dijkstra với phép ghép `max`; Dijkstra đúng cho mọi phép ghép đơn điệu không giảm (`f(d, w) ≥ d`) và đơn điệu theo `d`.

## 11. Phân tích Time/Space

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử từng `t` | `O(n⁴)` | `O(n²)` | `n²` lần BFS |
| 2. Nhị phân + BFS | `O(n² log n)` | `O(n²)` | `≈ 12` lần BFS khi `n = 50` |
| 3. Dijkstra / loang heap | `O(n² log n)` | `O(n²)` | thường nhanh nhất vì dừng sớm |
| 4. Union-Find | `O(n² α(n²))` | `O(n²)` | cần độ cao khác nhau để khỏi sort |

Với `n = 50`, `n² = 2500` — mọi cách trừ brute force đều tức thì. Ta đo nhanh:

In [19]:
import random, time

def random_grid(n, seed):
    rnd = random.Random(seed)
    vals = list(range(n * n))
    rnd.shuffle(vals)
    return [vals[i * n:(i + 1) * n] for i in range(n)]

big = random_grid(50, 1)
for name, f in [("binary", swim_binary), ("dijkstra", swim_dijkstra), ("dsu", swim_dsu),
                ("Solution", lambda g: Solution().swimInWater(g))]:
    t0 = time.perf_counter()
    res = f(big)
    print(f"{name:9}: {res}  ({(time.perf_counter() - t0) * 1000:.1f} ms)")

binary   : 2051  (37.8 ms)
dijkstra : 2051  (6.0 ms)
dsu      : 2051  (6.2 ms)
Solution : 2051  (2.8 ms)


## 12. Bộ test

In [20]:
methods = {
    "brute": swim_brute,
    "binary": swim_binary,
    "dijkstra": swim_dijkstra,
    "dsu": swim_dsu,
    "SolutionBinary": lambda g: SolutionBinary().swimInWater(g),
    "SolutionDijkstra": lambda g: SolutionDijkstra().swimInWater(g),
    "SolutionDSU": lambda g: SolutionDSU().swimInWater(g),
    "Solution": lambda g: Solution().swimInWater(g),
}

# 12.1 Ví dụ đề + edge cases
edge = EXAMPLES + [
    ([[0]], 0),                              # n = 1
    ([[0, 1], [2, 3]], 3),                   # đích cao nhất
    ([[3, 0], [1, 2]], 3),                   # đầu cao nhất
    ([[0, 3], [2, 1]], 2),                   # đi xuống rồi sang phải
    ([[0, 8, 7], [1, 6, 5], [2, 3, 4]], 4),  # đi vòng chữ U
]
for name, f in methods.items():
    for g, exp in edge:
        assert f(g) == exp, (name, g, f(g), exp)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed: mọi cách khớp brute force
for seed in range(300):
    n = random.Random(seed).randint(1, 6)
    g = random_grid(n, seed)
    ref = swim_brute(g)
    for name, f in methods.items():
        assert f(g) == ref, (name, g)
print("300 lưới ngẫu nhiên: mọi cách khớp brute force")

# 12.3 Lưới lớn: các cách nhanh khớp nhau
for seed in range(5):
    g = random_grid(50, 100 + seed)
    vals = {name: f(g) for name, f in methods.items() if name != "brute"}
    assert len(set(vals.values())) == 1, vals
print("Lưới 50x50: các cách nhanh khớp nhau")

Ví dụ + edge cases: OK


300 lưới ngẫu nhiên: mọi cách khớp brute force


Lưới 50x50: các cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên ô xuất phát cũng phải ngập

**Code sai:**

In [21]:
def wrong_start(grid):
    n = len(grid)
    seen = {(0, 0)}
    heap = []
    for nr, nc in neighbors(0, 0, n):           # SAI: không tính grid[0][0]
        seen.add((nr, nc)); heapq.heappush(heap, (grid[nr][nc], nr, nc))
    ans = 0
    while heap:
        h, r, c = heapq.heappop(heap)
        ans = max(ans, h)
        if (r, c) == (n - 1, n - 1):
            return ans
        for nr, nc in neighbors(r, c, n):
            if (nr, nc) not in seen:
                seen.add((nr, nc)); heapq.heappush(heap, (grid[nr][nc], nr, nc))
    return ans   # với n = 1 trả 0 dù grid[0][0] có thể > 0

g_bad = [[3, 0], [1, 2]]
print("sai:", wrong_start(g_bad), " đúng:", Solution().swimInWater(g_bad))

sai: 2  đúng: 3


**Vì sao sai:** đề yêu cầu *cả hai ô* đều `≤ t`, nên ngay ô `(0,0)` cũng phải ngập. Với `[[3,0],[1,2]]`, đáp án là 3 chứ không phải 2.

**Code đúng:** khởi tạo heap bằng `(grid[0][0], 0, 0)` (như `Solution`), hoặc `ans = grid[0][0]`.

### Lỗi 2 — Dùng BFS thường (đếm số bước) thay vì minimax

**Code sai:**

In [22]:
def wrong_bfs_steps(grid):
    """Đi đường ngắn nhất theo số bước rồi lấy max trên đường đó."""
    n = len(grid)
    parent = {(0, 0): None}
    q = deque([(0, 0)])
    while q:
        u = q.popleft()
        for v in neighbors(*u, n):
            if v not in parent:
                parent[v] = u; q.append(v)
    cur, best = (n - 1, n - 1), 0
    while cur:
        best = max(best, grid[cur[0]][cur[1]]); cur = parent[cur]
    return best

g_u = [[0, 1, 2], [7, 8, 3], [6, 5, 4]]
print("sai:", wrong_bfs_steps(g_u), " đúng:", Solution().swimInWater(g_u))

sai: 7  đúng: 4


**Vì sao sai:** đường ngắn nhất về số bước có thể phải trèo qua ô rất cao; đường vòng dài hơn lại thấp hơn. Bài này không quan tâm số bước, chỉ quan tâm **độ cao lớn nhất**.

**Code đúng:** heap theo độ cao (Dijkstra/Prim) hoặc nhị phân trên `t`.

### Lỗi 3 — Nhị phân với cận sai / vòng lặp vô hạn

**Code sai:**

In [23]:
def wrong_binary(grid, max_iter=50):
    n = len(grid)
    lo, hi = 0, n * n - 1
    it = 0
    while lo < hi and it < max_iter:
        it += 1
        mid = (lo + hi) // 2
        if can_reach(grid, mid):
            hi = mid - 1          # SAI: có thể bỏ mất đáp án mid
        else:
            lo = mid + 1
    return lo

for g in [g_u, g3] + [e[0] for e in EXAMPLES]:
    print("sai:", wrong_binary(g), " đúng:", swim_binary(g))

sai: 3  đúng: 4
sai: 7  đúng: 7
sai: 3  đúng: 3
sai: 16  đúng: 16
sai: 0  đúng: 0
sai: 3  đúng: 3


**Vì sao sai:** khi `can(mid)` đúng, `mid` *có thể chính là* đáp án; đặt `hi = mid - 1` làm mất nó. Mẫu "tìm phần tử đúng đầu tiên" phải là `hi = mid` khi đúng, `lo = mid + 1` khi sai, và điều kiện `while lo < hi`.

**Code đúng:** `swim_binary` ở mục 7.1.

### Lỗi 4 — Union-Find gộp với láng giềng chưa ngập

Nếu ở bước `t` ta gộp ô `pos[t]` với **mọi** láng giềng (kể cả ô cao hơn `t`), ta đã "đi xuyên" qua ô khô → đáp án quá nhỏ. Luôn kiểm tra `grid[nr][nc] < t` (hoặc `≤ t`).

In [24]:
def wrong_dsu(grid):
    n = len(grid); N = n * n
    pos = {grid[r][c]: (r, c) for r in range(n) for c in range(n)}
    dsu = DSU(N)
    for t in range(N):
        r, c = pos[t]
        for nr, nc in neighbors(r, c, n):
            dsu.union(r * n + c, nr * n + nc)        # SAI: không kiểm tra đã ngập
        if dsu.find(0) == dsu.find(N - 1):
            return t
    return N - 1

print("sai:", wrong_dsu(g3), " đúng:", swim_dsu(g3))

sai: 3  đúng: 7


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu "minimax path":** "thời điểm sớm nhất mà đi được", "nỗ lực tối thiểu", "trọng số lớn nhất trên đường càng nhỏ càng tốt". Ba công cụ: **nhị phân + BFS**, **Dijkstra với `max`**, **Union-Find/Kruskal**.
- Đáp án luôn `≥ max(grid[0][0], grid[n-1][n-1])` → dùng làm `lo` cho nhị phân.
- Nếu giá trị là hoán vị `0..n²-1` → có thể lập `pos[value]` để khỏi sort (Union-Find tuyến tính).
- Dijkstra "max" có thể dừng ngay khi **lấy ra** đích (không phải khi đẩy vào — tuy với bản gọn ở bài này hai cách trùng nhau vì ta lấy max với độ cao ô).
- Cùng họ: **1631. Path With Minimum Effort** (trọng số cạnh = chênh lệch độ cao), **1102. Path With Maximum Minimum Value** (maximin), **1970. Last Day Where You Can Still Cross** (nhị phân/Union-Find theo thời gian).

## 15. Biến thể

1. **Maximin (1102):** tìm đường có **giá trị nhỏ nhất trên đường** là **lớn nhất** → heap max (đẩy `-value`), hoặc DSU bật ô theo giá trị **giảm** dần.
2. **Trọng số trên cạnh (1631):** chi phí cạnh `|h[u] - h[v]|`, đường minimax → Dijkstra với `max(d, w)`.
3. **Độ cao không phân biệt:** Union-Find phải **sắp xếp** ô theo độ cao (`O(n² log n)`); các cách khác không đổi.
4. **Đi 8 hướng:** chỉ đổi `DIRS`.
5. **In ra đường đi tối ưu:** lưu `parent` trong Dijkstra rồi truy ngược.

Minh hoạ biến thể 1 (maximin) bằng heap max:

In [25]:
def maximin_path(grid):
    n = len(grid)
    seen = {(0, 0)}
    heap = [(-grid[0][0], 0, 0)]
    ans = grid[0][0]
    while heap:
        h, r, c = heapq.heappop(heap)
        ans = min(ans, -h)
        if (r, c) == (n - 1, n - 1):
            return ans
        for nr, nc in neighbors(r, c, n):
            if (nr, nc) not in seen:
                seen.add((nr, nc)); heapq.heappush(heap, (-grid[nr][nc], nr, nc))
    return ans

print(maximin_path([[5, 4, 5], [1, 2, 6], [7, 4, 6]]))   # LeetCode 1102 ví dụ 1 -> 4

4


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng `t` | BFS cho mỗi `t` | `O(n⁴)` | `O(n²)` | kiểm tra/đối chiếu |
| 2. Nhị phân + BFS | vị từ đơn điệu | `O(n² log n)` | `O(n²)` | khi viết BFS quen tay; tổng quát cho mọi điều kiện đơn điệu |
| 3. Dijkstra / Prim | heap theo độ cao, `ans = max` | `O(n² log n)` | `O(n²)` | **lựa chọn mặc định**, ngắn gọn |
| 4. Union-Find | bật ô theo độ cao tăng | `O(n² α)` | `O(n²)` | khi giá trị là hoán vị / khi nhiều truy vấn theo thời gian |

## 17. Bài tập

**Bài 1 (đoán output).** `swimInWater([[0,1,2],[5,4,3],[6,7,8]])` = ? và `swimInWater([[8,1],[2,0]])` = ?

**Bài 2 (tính tay).** Với lưới `[[0,6,4],[2,7,3],[1,5,8]]`... thử chạy nhị phân bằng tay: liệt kê các `mid` được thử và kết quả `can_reach`.

**Bài 3 (điền TODO).** Hoàn thiện `swim_path(grid)` trả về **danh sách ô** trên một đường tối ưu (dùng Dijkstra + `parent`).

**Bài 4 (tự viết lại).** Viết lại cách 4 nhưng **không** dùng giả thiết hoán vị: sắp xếp các ô theo độ cao.

**Bài 5 (mini challenge).** LeetCode 1631 — *Path With Minimum Effort*: chi phí đường đi là `max |h[u]-h[v]|` trên các cạnh. Viết `min_effort(heights)`.

In [26]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def swim_path(grid):
    # TODO: Dijkstra với dist = max; lưu parent[v] = u khi cập nhật; truy ngược từ đích
    pass

# Bài 4 — TODO
def swim_dsu_sorted(grid):
    # TODO: cells = sorted((grid[r][c], r, c) ...); bật lần lượt; gộp với ô đã bật
    pass

# Bài 5 — TODO
def min_effort(heights):
    # TODO: Dijkstra; cand = max(d, abs(h[u] - h[v]))
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: đáp án `≥ max(góc đầu, góc cuối)`.
- Mức 2: lưới 1: góc cuối là 8 — giá trị lớn nhất cả lưới → đáp án 8.
- Mức 3: lưới 2: `[[8,1],[2,0]]` — ô đầu cao 8 → phải đợi tới 8.

**Bài 2.**
- Mức 1: `lo = max(0, 8) = 8`, `hi = 8`.
- Mức 2: vòng lặp không chạy lần nào.
- Mức 3: đáp án 8 (ô đích là lớn nhất).

**Bài 3.**
- Mức 1: `parent[(0,0)] = None`.
- Mức 2: cập nhật `parent[v] = u` mỗi khi `dist[v]` giảm.
- Mức 3: truy ngược từ `(n-1,n-1)`, đảo ngược danh sách; kiểm tra `max` trên đường bằng đáp án.

**Bài 4.**
- Mức 1: `active[r][c] = True` khi bật.
- Mức 2: gộp với láng giềng `active`.
- Mức 3: trả về độ cao của ô vừa bật khi đầu–đích nối.

**Bài 5.**
- Mức 1: giống cách 3 bản chuẩn, `dist[0][0] = 0`.
- Mức 2: trọng số cạnh `abs(h[r][c] - h[nr][nc])`.
- Mức 3: dừng khi lấy ra đích khỏi heap.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().swimInWater([[0, 1, 2], [5, 4, 3], [6, 7, 8]]) == 8
assert Solution().swimInWater([[8, 1], [2, 0]]) == 8

# Bài 2
g_b2 = [[0, 6, 4], [2, 7, 3], [1, 5, 8]]
assert swim_brute(g_b2) == 8

# Bài 3
def swim_path_ans(grid):
    n = len(grid)
    INF = float('inf')
    dist = {(0, 0): grid[0][0]}
    parent = {(0, 0): None}
    heap = [(grid[0][0], 0, 0)]
    while heap:
        d, r, c = heapq.heappop(heap)
        if d > dist[(r, c)]:
            continue
        if (r, c) == (n - 1, n - 1):
            break
        for v in neighbors(r, c, n):
            cand = max(d, grid[v[0]][v[1]])
            if cand < dist.get(v, INF):
                dist[v] = cand; parent[v] = (r, c)
                heapq.heappush(heap, (cand, v[0], v[1]))
    path, cur = [], (n - 1, n - 1)
    while cur is not None:
        path.append(cur); cur = parent[cur]
    return path[::-1]

for seed in range(50):
    g = random_grid(random.Random(seed).randint(1, 6), seed)
    p = swim_path_ans(g)
    assert p[0] == (0, 0) and p[-1] == (len(g) - 1, len(g) - 1)
    assert all(abs(a[0] - b[0]) + abs(a[1] - b[1]) == 1 for a, b in zip(p, p[1:]))
    assert max(g[r][c] for r, c in p) == swim_brute(g)

# Bài 4
def swim_dsu_sorted_ans(grid):
    n = len(grid)
    cells = sorted((grid[r][c], r, c) for r in range(n) for c in range(n))
    active = [[False] * n for _ in range(n)]
    dsu = DSU(n * n)
    for h, r, c in cells:
        active[r][c] = True
        for nr, nc in neighbors(r, c, n):
            if active[nr][nc]:
                dsu.union(r * n + c, nr * n + nc)
        if active[0][0] and active[n - 1][n - 1] and dsu.find(0) == dsu.find(n * n - 1):
            return h
    return cells[-1][0]

for seed in range(100):
    g = random_grid(random.Random(seed).randint(1, 6), seed)
    assert swim_dsu_sorted_ans(g) == swim_brute(g)
# lưới có độ cao trùng nhau
assert swim_dsu_sorted_ans([[1, 1], [5, 1]]) == 1

# Bài 5
def min_effort_ans(heights):
    R, C = len(heights), len(heights[0])
    dist = [[float('inf')] * C for _ in range(R)]
    dist[0][0] = 0
    heap = [(0, 0, 0)]
    while heap:
        d, r, c = heapq.heappop(heap)
        if d > dist[r][c]:
            continue
        if (r, c) == (R - 1, C - 1):
            return d
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C:
                cand = max(d, abs(heights[r][c] - heights[nr][nc]))
                if cand < dist[nr][nc]:
                    dist[nr][nc] = cand
                    heapq.heappush(heap, (cand, nr, nc))
    return 0

assert min_effort_ans([[1, 2, 2], [3, 8, 2], [5, 3, 5]]) == 2
assert min_effort_ans([[1, 2, 3], [3, 8, 4], [5, 3, 5]]) == 1
assert min_effort_ans([[1, 2, 1, 1, 1], [1, 2, 1, 2, 1], [1, 2, 1, 2, 1], [1, 2, 1, 2, 1], [1, 1, 1, 2, 1]]) == 0
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "loang theo ô thấp nhất" (minimax trên lưới đỉnh):**

```python
seen = [[False]*n for _ in range(n)]; seen[0][0] = True
heap = [(grid[0][0], 0, 0)]; ans = 0
while heap:
    h, r, c = heappop(heap); ans = max(ans, h)
    if (r, c) == (n-1, n-1): return ans
    for nr, nc in 4 láng giềng hợp lệ chưa seen:
        seen[nr][nc] = True; heappush(heap, (grid[nr][nc], nr, nc))
```

**Template nhị phân trên đáp án:**

```python
lo, hi = cận_dưới, cận_trên
while lo < hi:
    mid = (lo + hi) // 2
    if ok(mid): hi = mid
    else: lo = mid + 1
return lo
```

**Checklist trước khi submit:**

- [ ] Ô xuất phát `grid[0][0]` được tính vào đáp án.
- [ ] Trường hợp `n = 1` trả `grid[0][0]`.
- [ ] Kiểm tra biên `0 <= nr < n and 0 <= nc < n`.
- [ ] Heap theo **độ cao**, không theo số bước.
- [ ] Nhị phân: `hi = mid` khi đúng, `lo = mid + 1` khi sai.
- [ ] Union-Find: chỉ gộp với láng giềng **đã ngập**.